# Lab: Multi-Sensor Fusion with Asynchronous GPS + IMU
**State Estimation and Localization for Self-Driving Cars**

---

## 🎯 Learning Objectives
1. **Master Asynchronous Multi-Rate Estimation**: Understand why high-rate IMU (100 Hz) drives kinematic **Prediction** while low-rate GPS (10 Hz) drives **Correction**.
2. **Deconstruct Covariances ($Q$ vs $R$)**:
   - Why IMU sensor noise is modeled as **Process Noise ($Q_{\text{imu}}$)**.
   - Why GPS uncertainty is modeled as **Measurement Noise ($R_{\text{gps}}$)**.
3. **Online Sensor Bias Calibration**: Track how the filter estimates accelerometer bias ($b_a$) and gyroscope bias ($b_g$) in real time.
4. **Unified Outage Handling**: Observe how the filter seamlessly traverses a **10-second GPS tunnel blackout** without changing models, and how the $3\sigma$ covariance envelope bounds the drift.

## 📐 Mathematical Formulation

### 1. State Vector Definition
We track the vehicle in a 2D navigation frame with an 8-dimensional state vector:

$$\mathbf{x} = \begin{bmatrix} p_x \\ p_y \\ v_x \\ v_y \\ \theta \\ b_{ax} \\ b_{ay} \\ b_g \end{bmatrix} \in \mathbb{R}^8$$

where:
- $\mathbf{p} = [p_x, p_y]^T$: 2D Position (meters)
- $\mathbf{v} = [v_x, v_y]^T$: 2D Velocity (m/s)
- $\theta$: Vehicle heading / yaw angle (radians)
- $\mathbf{b}_a = [b_{ax}, b_{ay}]^T$: Accelerometer biases ($\text{m/s}^2$)
- $b_g$: Gyroscope bias (rad/s)

---

### 2. High-Rate Prediction Step (IMU @ 100 Hz, $\Delta t = 10\text{ ms}$)
Given raw IMU readings $\mathbf{a}_m = [a_{mx}, a_{my}]^T$ and $\omega_m$:

$$\hat{\mathbf{a}}_k = \mathbf{R}(\hat{\theta}_{k-1}) (\mathbf{a}_{m,k} - \hat{\mathbf{b}}_{a,k-1})$$
$$\hat{\mathbf{p}}_k = \hat{\mathbf{p}}_{k-1} + \hat{\mathbf{v}}_{k-1} \Delta t + \frac{1}{2} \hat{\mathbf{a}}_k \Delta t^2$$
$$\hat{\mathbf{v}}_k = \hat{\mathbf{v}}_{k-1} + \hat{\mathbf{a}}_k \Delta t$$
$$\hat{\theta}_k = \hat{\theta}_{k-1} + (\omega_{m,k} - \hat{b}_{g,k-1}) \Delta t$$

The error-state transition Jacobian $\mathbf{F}_{k-1} \in \mathbb{R}^{8 \times 8}$ is:

$$\mathbf{F}_{k-1} = \begin{bmatrix}
\mathbf{I}_2 & \mathbf{I}_2 \Delta t & \mathbf{0}_{2\times 1} & \mathbf{0}_{2\times 2} & \mathbf{0}_{2\times 1} \\
\mathbf{0}_2 & \mathbf{I}_2 & \begin{bmatrix} -\sin\hat{\theta} a_x - \cos\hat{\theta} a_y \\ \cos\hat{\theta} a_x - \sin\hat{\theta} a_y \end{bmatrix} \Delta t & -\mathbf{R}(\hat{\theta}) \Delta t & \mathbf{0}_{2\times 1} \\
\mathbf{0}_{1\times 2} & \mathbf{0}_{1\times 2} & 1 & \mathbf{0}_{1\times 2} & -\Delta t \\
\mathbf{0}_{2\times 2} & \mathbf{0}_{2\times 2} & \mathbf{0}_{2\times 1} & \mathbf{I}_2 & \mathbf{0}_{2\times 1} \\
\mathbf{0}_{1\times 2} & \mathbf{0}_{1\times 2} & 0 & \mathbf{0}_{1\times 2} & 1
\end{bmatrix}$$

$$\mathbf{P}_{k|k-1} = \mathbf{F}_{k-1} \mathbf{P}_{k-1|k-1} \mathbf{F}_{k-1}^T + \mathbf{Q}_{\text{imu}}$$

---

### 3. Low-Rate Correction Step (GPS @ 10 Hz, $\Delta t = 100\text{ ms}$)
When a GPS fix $\mathbf{z}_{\text{gps}} = [x_{\text{gps}}, y_{\text{gps}}]^T$ arrives:

$$\mathbf{H} = \begin{bmatrix} 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 \end{bmatrix} \in \mathbb{R}^{2 \times 8}, \quad \mathbf{R}_{\text{gps}} = \begin{bmatrix} \sigma_{\text{gps}}^2 & 0 \\ 0 & \sigma_{\text{gps}}^2 \end{bmatrix}$$

$$\mathbf{y}_k = \mathbf{z}_{\text{gps}} - \mathbf{H} \hat{\mathbf{x}}_{k|k-1}$$
$$\mathbf{S}_k = \mathbf{H} \mathbf{P}_{k|k-1} \mathbf{H}^T + \mathbf{R}_{\text{gps}}$$
$$\mathbf{K}_k = \mathbf{P}_{k|k-1} \mathbf{H}^T \mathbf{S}_k^{-1} \in \mathbb{R}^{8 \times 2}$$
$$\hat{\mathbf{x}}_{k|k} = \hat{\mathbf{x}}_{k|k-1} + \mathbf{K}_k \mathbf{y}_k$$
$$\mathbf{P}_{k|k} = (\mathbf{I}_8 - \mathbf{K}_k \mathbf{H}) \mathbf{P}_{k|k-1} (\mathbf{I}_8 - \mathbf{K}_k \mathbf{H})^T + \mathbf{K}_k \mathbf{R}_{\text{gps}} \mathbf{K}_k^T \quad \text{(Joseph Form)}$$

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set reproducible seed
np.random.seed(42)
print("Interactive Plotly environment loaded successfully!")

## 🚗 1. Trajectory & Multi-Rate Sensor Simulation
We generate a 60-second curved roadway test track with:
- **100 Hz IMU**: Corrupted by acceleration noise ($\sigma_a = 0.15\text{ m/s}^2$), gyro noise ($\sigma_g = 0.008\text{ rad/s}$), and persistent hardware biases ($b_{ax} = +0.20\text{ m/s}^2, b_{ay} = -0.15\text{ m/s}^2, b_g = +0.025\text{ rad/s}$).
- **10 Hz GPS**: Position fixes corrupted by Gaussian noise ($\sigma_{\text{gps}} = 1.5\text{ m}$).
- **10-Second Tunnel Blackout**: GPS drops completely between $t = 25.0\text{ s}$ and $t = 35.0\text{ s}$.

In [ ]:
# Simulation Time Settings
dt_imu = 0.01   # 100 Hz IMU rate (10 ms)
dt_gps = 0.10   # 10 Hz GPS rate (100 ms)
duration = 60.0 # 60 seconds total runtime
t = np.arange(0, duration, dt_imu)
N = len(t)

# Ground Truth Figure-8 Trajectory
speed = 14.0    # 14 m/s (~50 km/h)
radius = 50.0   # 50m radius loop
omega_turn = speed / radius

gt_px = radius * np.sin(omega_turn * t)
gt_py = radius * np.sin(2.0 * omega_turn * t) / 2.0

gt_vx = radius * omega_turn * np.cos(omega_turn * t)
gt_vy = radius * omega_turn * np.cos(2.0 * omega_turn * t)

gt_ax = -radius * (omega_turn**2) * np.sin(omega_turn * t)
gt_ay = -2.0 * radius * (omega_turn**2) * np.sin(2.0 * omega_turn * t)

gt_theta = np.arctan2(gt_vy, gt_vx)
gt_omega = np.gradient(gt_theta, dt_imu)

# Ground Truth Sensor Biases
true_b_ax = 0.20   # m/s^2 (uncompensated acceleration offset)
true_b_ay = -0.15  # m/s^2
true_b_g  = 0.025  # rad/s (~1.4 deg/s gyro drift rate)

# Sensor Noise Covariance Parameters
sigma_a = 0.15     # Accelerometer noise std (m/s^2)
sigma_g = 0.008    # Gyroscope noise std (rad/s)
sigma_gps = 1.5    # GPS horizontal fix noise std (meters)

# 1. Generate IMU Specific Force & Angular Velocity in Vehicle Body Frame
imu_ax = np.zeros(N)
imu_ay = np.zeros(N)
imu_gz = gt_omega + true_b_g + np.random.normal(0, sigma_g, N)

for k in range(N):
    c, s = np.cos(gt_theta[k]), np.sin(gt_theta[k])
    R_nb = np.array([[c, -s], [s, c]]) # Body-to-Navigation rotation
    a_body = R_nb.T @ np.array([gt_ax[k], gt_ay[k]])
    imu_ax[k] = a_body[0] + true_b_ax + np.random.normal(0, sigma_a)
    imu_ay[k] = a_body[1] + true_b_ay + np.random.normal(0, sigma_a)

# 2. Generate GPS Fixes at 10 Hz with 10-Second Tunnel Outage
tunnel_start = 25.0 # seconds
tunnel_end   = 35.0 # seconds

gps_times = []
gps_x = []
gps_y = []

for k in range(N):
    current_t = t[k]
    if k % int(dt_gps / dt_imu) == 0:
        # Check if satellite reception is blocked
        if not (tunnel_start <= current_t <= tunnel_end):
            gps_times.append(current_t)
            gps_x.append(gt_px[k] + np.random.normal(0, sigma_gps))
            gps_y.append(gt_py[k] + np.random.normal(0, sigma_gps))

gps_times = np.array(gps_times)
gps_x = np.array(gps_x)
gps_y = np.array(gps_y)

print(f"Generated {N} IMU samples (100 Hz) and {len(gps_times)} GPS fixes (10 Hz, 10s tunnel outage).")

## 🛠️ 2. The Multi-Rate Sensor Fusion Engine Implementation

In [ ]:
class GPS_IMU_Fusion_2D:
    """
    Asynchronous 2D Error-State Extended Kalman Filter.
    State: x = [p_x, p_y, v_x, v_y, theta, b_ax, b_ay, b_g]^T (8x1)
    """
    def __init__(self, x0, P0, q_accel, q_gyro, q_bias_a, q_bias_g, r_gps):
        self.x = np.array(x0, dtype=float).reshape(8, 1)
        self.P = np.array(P0, dtype=float)
        
        self.q_accel = q_accel   # Accelerometer noise variance
        self.q_gyro = q_gyro     # Gyroscope noise variance
        self.q_bias_a = q_bias_a # Accelerometer bias random walk variance
        self.q_bias_g = q_bias_g # Gyroscope bias random walk variance
        self.r_gps = r_gps       # GPS measurement noise variance
        
    def predict(self, a_body_meas, gyro_meas, dt):
        """High-rate prediction step (100 Hz): Propagate state & covariance"""
        # 1. Current State Estimates
        vx, vy = self.x[2, 0], self.x[3, 0]
        theta  = self.x[4, 0]
        b_ax, b_ay = self.x[5, 0], self.x[6, 0]
        b_g    = self.x[7, 0]
        
        # 2. Subtract Current Bias Estimates
        ax_unbiased = a_body_meas[0] - b_ax
        ay_unbiased = a_body_meas[1] - b_ay
        omega_unbiased = gyro_meas - b_g
        
        # 3. Rotate Specific Force from Body to Navigation Frame
        c, s = np.cos(theta), np.sin(theta)
        R_nb = np.array([[c, -s], [s, c]])
        a_nav = R_nb @ np.array([ax_unbiased, ay_unbiased])
        
        # 4. Propagate Kinematics Forward
        self.x[0, 0] += vx * dt + 0.5 * a_nav[0] * dt**2
        self.x[1, 0] += vy * dt + 0.5 * a_nav[1] * dt**2
        self.x[2, 0] += a_nav[0] * dt
        self.x[3, 0] += a_nav[1] * dt
        self.x[4, 0] = (theta + omega_unbiased * dt + np.pi) % (2 * np.pi) - np.pi # Wrap [-pi, pi]
        
        # 5. Error State Transition Jacobian F (8x8)
        F = np.eye(8)
        F[0:2, 2:4] = np.eye(2) * dt
        
        # Velocity-Attitude Coupling
        F[2, 4] = (-s * ax_unbiased - c * ay_unbiased) * dt
        F[3, 4] = ( c * ax_unbiased - s * ay_unbiased) * dt
        
        # Sensor Bias Coupling
        F[2:4, 5:7] = -R_nb * dt
        F[4, 7] = -dt
        
        # 6. Process Noise Covariance Q (8x8)
        Q = np.zeros((8, 8))
        Q[0:2, 0:2] = np.eye(2) * (0.25 * dt**4 * self.q_accel)
        Q[2:4, 2:4] = np.eye(2) * (dt**2 * self.q_accel)
        Q[4, 4]     = dt**2 * self.q_gyro
        Q[5:7, 5:7] = np.eye(2) * (dt * self.q_bias_a)
        Q[7, 7]     = dt * self.q_bias_g
        
        # 7. Covariance Propagation
        self.P = F @ self.P @ F.T + Q

    def update_gps(self, z_gps, r_override=None):
        """Low-rate measurement update (10 Hz): Correct state & contract covariance"""
        R = np.eye(2) * (r_override if r_override is not None else self.r_gps)
        
        # Observation matrix H: observes position [px, py]
        H = np.zeros((2, 8))
        H[0, 0] = 1.0
        H[1, 1] = 1.0
        
        # Innovation
        z = np.array(z_gps, dtype=float).reshape(2, 1)
        y = z - H @ self.x
        
        # Innovation Covariance
        S = H @ self.P @ H.T + R
        
        # Kalman Gain (8x2)
        K = self.P @ H.T @ np.linalg.inv(S)
        
        # State Correction (updates pose, velocity, and recalibrates biases!)
        self.x += K @ y
        self.x[4, 0] = (self.x[4, 0] + np.pi) % (2 * np.pi) - np.pi
        
        # Covariance Update (Joseph form for guaranteed positive semi-definiteness)
        I = np.eye(8)
        self.P = (I - K @ H) @ self.P @ (I - K @ H).T + K @ R @ K.T
        
        # Normalized Innovation Squared (NIS)
        nis = float((y.T @ np.linalg.inv(S) @ y).item())
        return y, S, nis

print("GPS_IMU_Fusion_2D initialized.")

## 🔄 3. Simulation Execution: Fusing Asynchronous Streams

In [ ]:
# Initial State & Covariance
x0 = [gt_px[0], gt_py[0], gt_vx[0], gt_vy[0], gt_theta[0], 0.0, 0.0, 0.0]
P0 = np.diag([1.0, 1.0, 0.5, 0.5, 0.05, 0.1, 0.1, 0.01])

ekf = GPS_IMU_Fusion_2D(
    x0=x0,
    P0=P0,
    q_accel=sigma_a**2,
    q_gyro=sigma_g**2,
    q_bias_a=1e-4,
    q_bias_g=1e-5,
    r_gps=sigma_gps**2
)

# Pure IMU Dead Reckoning baseline (no GPS corrections)
imu_only_x = np.array(x0, dtype=float).reshape(8, 1)

# Data Logs
est_p = np.zeros((N, 2))
est_v = np.zeros((N, 2))
est_theta = np.zeros(N)
est_b_a = np.zeros((N, 2))
est_b_g = np.zeros(N)
est_cov_p = np.zeros((N, 2))
dead_reckoning_p = np.zeros((N, 2))
nis_history = []
nis_timestamps = []

gps_ptr = 0

for k in range(N):
    current_t = t[k]
    a_meas = [imu_ax[k], imu_ay[k]]
    g_meas = imu_gz[k]
    
    # 1. 100 Hz IMU Prediction
    ekf.predict(a_meas, g_meas, dt_imu)
    
    # Pure IMU Dead Reckoning update
    c_dr, s_dr = np.cos(imu_only_x[4, 0]), np.sin(imu_only_x[4, 0])
    R_dr = np.array([[c_dr, -s_dr], [s_dr, c_dr]])
    a_dr = R_dr @ np.array(a_meas)
    imu_only_x[0, 0] += imu_only_x[2, 0] * dt_imu + 0.5 * a_dr[0] * dt_imu**2
    imu_only_x[1, 0] += imu_only_x[3, 0] * dt_imu + 0.5 * a_dr[1] * dt_imu**2
    imu_only_x[2, 0] += a_dr[0] * dt_imu
    imu_only_x[3, 0] += a_dr[1] * dt_imu
    imu_only_x[4, 0] += g_meas * dt_imu
    dead_reckoning_p[k] = [imu_only_x[0, 0], imu_only_x[1, 0]]
    
    # 2. Check for GPS Arrival (10 Hz)
    if gps_ptr < len(gps_times) and abs(current_t - gps_times[gps_ptr]) < 1e-4:
        z_gps = [gps_x[gps_ptr], gps_y[gps_ptr]]
        _, _, nis = ekf.update_gps(z_gps)
        nis_history.append(nis)
        nis_timestamps.append(current_t)
        gps_ptr += 1
        
    # Save state logs
    est_p[k] = [ekf.x[0, 0], ekf.x[1, 0]]
    est_v[k] = [ekf.x[2, 0], ekf.x[3, 0]]
    est_theta[k] = ekf.x[4, 0]
    est_b_a[k] = [ekf.x[5, 0], ekf.x[6, 0]]
    est_b_g[k] = ekf.x[7, 0]
    est_cov_p[k] = [np.sqrt(ekf.P[0, 0]), np.sqrt(ekf.P[1, 1])]

pos_errors = np.linalg.norm(est_p - np.column_stack([gt_px, gt_py]), axis=1)
dr_errors  = np.linalg.norm(dead_reckoning_p - np.column_stack([gt_px, gt_py]), axis=1)

print(f"Simulation Completed Successfully!")
print(f"Fused EKF Position RMSE: {np.sqrt(np.mean(pos_errors**2)):.3f} m")
print(f"Pure IMU Dead Reckoning Final Error: {dr_errors[-1]:.2f} m (Drift Explosion)")
print(f"Online Estimated Gyro Bias: {est_b_g[-1]:.4f} rad/s (True: {true_b_g:.4f})")
print(f"Online Estimated Accel Bias X: {est_b_a[-1, 0]:.4f} m/s^2 (True: {true_b_ax:.4f})")

## 📊 4. Interactive Plotly Visualizations
Explore the interactive plots below to observe how sensor fusion bridges GPS outages and converges sensor biases.

In [ ]:
# ---------------------------------------------------------------------------
# PLOT 1: 2D Interactive Trajectory Map
# ---------------------------------------------------------------------------
fig_traj = go.Figure()

fig_traj.add_trace(go.Scatter(
    x=gt_px, y=gt_py,
    mode='lines', name='Ground Truth Path',
    line=dict(color='black', width=3)
))

fig_traj.add_trace(go.Scatter(
    x=gps_x, y=gps_y,
    mode='markers', name='Noisy GPS Fixes (10 Hz)',
    marker=dict(color='red', size=4, opacity=0.6)
))

fig_traj.add_trace(go.Scatter(
    x=est_p[:, 0], y=est_p[:, 1],
    mode='lines', name='Fused ES-EKF (100 Hz)',
    line=dict(color='#0066FF', width=2.5)
))

# Highlight Tunnel Outage Segment
tunnel_mask = (t >= tunnel_start) & (t <= tunnel_end)
fig_traj.add_trace(go.Scatter(
    x=est_p[tunnel_mask, 0], y=est_p[tunnel_mask, 1],
    mode='lines', name='Tunnel Blackout (10s Dead Reckoning)',
    line=dict(color='orange', width=3.5, dash='dash')
))

fig_traj.add_trace(go.Scatter(
    x=dead_reckoning_p[:, 0], y=dead_reckoning_p[:, 1],
    mode='lines', name='Pure IMU Dead Reckoning (No GPS)',
    line=dict(color='green', width=1.5, dash='dot')
))

fig_traj.update_layout(
    title='<b>Multi-Sensor Vehicle Trajectory (GPS + IMU + 10s Tunnel Blackout)</b>',
    xaxis_title='East Position [m]', yaxis_title='North Position [m]',
    template='plotly_white',
    height=600,
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)')
)
fig_traj.show()

In [ ]:
# ---------------------------------------------------------------------------
# PLOT 2: Position Estimation Error vs 3-Sigma Covariance Bounds
# ---------------------------------------------------------------------------
fig_bounds = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=(
        '<b>East (X) Position Error vs ±3σ Uncertainty Bounds</b>',
        '<b>North (Y) Position Error vs ±3σ Uncertainty Bounds</b>'
    )
)

err_x = est_p[:, 0] - gt_px
err_y = est_p[:, 1] - gt_py
three_sigma_x = 3.0 * est_cov_p[:, 0]
three_sigma_y = 3.0 * est_cov_p[:, 1]

# Row 1: X Error
fig_bounds.add_trace(go.Scatter(x=t, y=err_x, mode='lines', name='Error X [m]', line=dict(color='blue')), row=1, col=1)
fig_bounds.add_trace(go.Scatter(x=t, y=three_sigma_x, mode='lines', name='+3σ Bound', line=dict(color='red', dash='dash')), row=1, col=1)
fig_bounds.add_trace(go.Scatter(x=t, y=-three_sigma_x, mode='lines', name='-3σ Bound', line=dict(color='red', dash='dash'), showlegend=False), row=1, col=1)

# Row 2: Y Error
fig_bounds.add_trace(go.Scatter(x=t, y=err_y, mode='lines', name='Error Y [m]', line=dict(color='purple')), row=2, col=1)
fig_bounds.add_trace(go.Scatter(x=t, y=three_sigma_y, mode='lines', name='+3σ Bound Y', line=dict(color='red', dash='dash')), row=2, col=1)
fig_bounds.add_trace(go.Scatter(x=t, y=-three_sigma_y, mode='lines', name='-3σ Bound Y', line=dict(color='red', dash='dash'), showlegend=False), row=2, col=1)

# Shaded Tunnel Outage Region on Both Panels
for r in [1, 2]:
    fig_bounds.add_vrect(
        x0=tunnel_start, x1=tunnel_end,
        fillcolor='gray', opacity=0.25,
        layer='below', line_width=0,
        annotation_text='GPS Tunnel Blackout (10s)' if r == 1 else '',
        annotation_position='top left',
        row=r, col=1
    )

fig_bounds.update_layout(
    height=600,
    template='plotly_white',
    xaxis2_title='Time [s]'
)
fig_bounds.show()

In [ ]:
# ---------------------------------------------------------------------------
# PLOT 3: Online Sensor Bias Calibration
# ---------------------------------------------------------------------------
fig_biases = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=(
        '<b>Gyroscope Bias Online Estimation (b_g)</b>',
        '<b>Accelerometer Bias Online Estimation (b_ax, b_ay)</b>'
    )
)

# Gyro Bias Panel
fig_biases.add_trace(go.Scatter(x=t, y=est_b_g, mode='lines', name='Estimated b_g', line=dict(color='#0066CC', width=2.5)), row=1, col=1)
fig_biases.add_trace(go.Scatter(x=[0, duration], y=[true_b_g, true_b_g], mode='lines', name='True b_g', line=dict(color='black', dash='dash', width=2)), row=1, col=1)

# Accel Bias Panel
fig_biases.add_trace(go.Scatter(x=t, y=est_b_a[:, 0], mode='lines', name='Estimated b_ax', line=dict(color='#E65100', width=2)), row=2, col=1)
fig_biases.add_trace(go.Scatter(x=[0, duration], y=[true_b_ax, true_b_ax], mode='lines', name='True b_ax', line=dict(color='#E65100', dash='dash')), row=2, col=1)
fig_biases.add_trace(go.Scatter(x=t, y=est_b_a[:, 1], mode='lines', name='Estimated b_ay', line=dict(color='#7B1FA2', width=2)), row=2, col=1)
fig_biases.add_trace(go.Scatter(x=[0, duration], y=[true_b_ay, true_b_ay], mode='lines', name='True b_ay', line=dict(color='#7B1FA2', dash='dash')), row=2, col=1)

for r in [1, 2]:
    fig_biases.add_vrect(
        x0=tunnel_start, x1=tunnel_end,
        fillcolor='gray', opacity=0.25,
        layer='below', line_width=0,
        row=r, col=1
    )

fig_biases.update_layout(
    height=600,
    template='plotly_white',
    yaxis_title='Bias [rad/s]',
    yaxis2_title='Bias [m/s²]',
    xaxis2_title='Time [s]'
)
fig_biases.show()

In [ ]:
# ---------------------------------------------------------------------------
# PLOT 4: Filter Statistical Consistency (Normalized Innovation Squared - NIS)
# ---------------------------------------------------------------------------
chi2_threshold_95 = 5.991  # 2-DOF 95% confidence limit

fig_nis = go.Figure()
fig_nis.add_trace(go.Scatter(
    x=nis_timestamps, y=nis_history,
    mode='markers+lines',
    name='GPS Innovation NIS (2-DOF)',
    marker=dict(color='teal', size=5),
    line=dict(color='teal', width=1)
))

fig_nis.add_trace(go.Scatter(
    x=[0, duration], y=[chi2_threshold_95, chi2_threshold_95],
    mode='lines',
    name='χ² 95% Confidence Threshold (5.99)',
    line=dict(color='red', dash='dash', width=2)
))

fig_nis.add_vrect(
    x0=tunnel_start, x1=tunnel_end,
    fillcolor='gray', opacity=0.25,
    layer='below', line_width=0,
    annotation_text='Tunnel: Zero GPS Innovations',
    annotation_position='top left'
)

fig_nis.update_layout(
    title='<b>Filter Consistency: Normalized Innovation Squared (NIS) vs χ² Bound</b>',
    xaxis_title='Time [s]',
    yaxis_title='NIS (y^T S^-1 y)',
    template='plotly_white',
    height=450
)
fig_nis.show()

## 🎓 Summary of Key Sensor Fusion Insights

| Dimension | Pure IMU (Dead Reckoning) | Pure GPS | Fused GPS + IMU (ES-EKF) |
| :--- | :--- | :--- | :--- |
| **Output Frequency** | 100 Hz | 10 Hz | **100 Hz continuous stream** |
| **Latency** | < 2 ms | 50 - 200 ms | **Instantaneous low latency** |
| **Drift Behavior** | **Explodes cubically** ($>100\text{ m}$) | Zero long-term drift | **Zero long-term drift (< 0.5m RMSE)** |
| **Tunnel Handling** | Unaware of error growth | Total blackout | **$3\sigma$ envelope accurately tracks uncertainty expansion & recovers instantly** |
| **Sensor Biases** | Destroys pose integration | Cannot observe biases | **Learns and eliminates $\mathbf{b}_a, b_g$ online** |